# Rina Park — Colab bootstrap

1. GPU 런타임 확인
2. Drive mount + Secrets (`HF_TOKEN`, `GIT_TOKEN`)
3. git clone/pull → `requirements-colab.txt` → symlink → HF `--tier sdxl`
4. (선택) smoke `generate_ig_quality.py`

자세한 설명: `rina_park/ops/COLAB_SETUP.md`

In [ ]:
# 0) GPU check
import torch
print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
assert torch.cuda.is_available(), 'Runtime → Change runtime type → GPU'

In [ ]:
# 1) Drive + secrets
from google.colab import drive, userdata
import os

drive.mount('/content/drive')

def _secret(name: str) -> str:
    try:
        return userdata.get(name) or ''
    except Exception:
        return ''

hf = _secret('HF_TOKEN')
git_token = _secret('GIT_TOKEN')
if hf:
    os.environ['HF_TOKEN'] = hf
    os.environ['HUGGING_FACE_HUB_TOKEN'] = hf
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['USE_TF'] = '0'
os.environ['USE_TORCH'] = '1'
print('HF_TOKEN:', 'yes' if hf else 'no')
print('GIT_TOKEN:', 'yes' if git_token else 'no')

In [ ]:
# 2) Clone or pull latest
from pathlib import Path
import shutil

REPO = Path('/content/ai_influencer')
url = 'https://github.com/pkang0831/ai_advertisement_brandings.git'
if git_token:
    url = f'https://{git_token}@github.com/pkang0831/ai_advertisement_brandings.git'

if REPO.exists() and not (REPO / 'rina_park').is_dir():
    shutil.rmtree(REPO)
if not (REPO / 'rina_park').is_dir():
    !git clone --depth 1 {url} {REPO}
else:
    print('already cloned — pulling', REPO)
    %cd /content/ai_influencer
    !git pull --ff-only || true

%cd /content/ai_influencer
!git rev-parse --short HEAD

In [ ]:
# 3) Pip pins (+ protobuf bump for Colab TF/protobuf clash)
import os
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['USE_TF'] = '0'
os.environ['USE_TORCH'] = '1'

!pip -q uninstall -y diffusers transformers accelerate huggingface_hub peft 2>/dev/null || true
!pip -q install --no-cache-dir -r requirements-colab.txt
!pip -q install --no-cache-dir --upgrade 'protobuf>=5.28.3'
print('pip ok')
import google.protobuf
print('protobuf', google.protobuf.__version__)

In [ ]:
# 4) Symlink Drive ↔ rina_park
!python rina_park/scripts/colab_bootstrap.py
import os, sys
sys.path[:0] = ['/content/ai_influencer', '/content/ai_influencer/rina_park']
os.environ['PYTHONPATH'] = '/content/ai_influencer:/content/ai_influencer/rina_park'

In [ ]:
# 5) HF download → Google Drive only (cache also on Drive)
# Prefer 'sdxl'. wan/qwen/all need --confirm-large and tens–hundreds of GiB.
import os
from pathlib import Path

DRIVE = Path('/content/drive/MyDrive/rina_park_colab')
assert DRIVE.exists(), 'Mount Google Drive first'
os.environ['HF_HOME'] = str(DRIVE / '.hf_home')
os.environ['HF_HUB_CACHE'] = str(DRIVE / '.hf_home' / 'hub')
os.environ['HUGGINGFACE_HUB_CACHE'] = os.environ['HF_HUB_CACHE']

TIER = 'sdxl'  # 'sdxl' | 'wan' | 'qwen_cuda' | 'all'
extra = '--confirm-large' if TIER != 'sdxl' else ''
!python rina_park/scripts/colab_download_hf_models.py --tier {TIER} \
  --models-root /content/drive/MyDrive/rina_park_colab/models \
  --hf-home /content/drive/MyDrive/rina_park_colab/.hf_home {extra}

# If /content disk is full from an earlier bad run:
# !rm -rf /root/.cache/huggingface ~/.cache/huggingface
!df -h /content | sed -n '1,2p'

In [ ]:
# 6) Smoke (needs RealVis + preferably character LoRA from Mac rclone)
import os
from pathlib import Path

os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['USE_TF'] = '0'
os.environ['USE_TORCH'] = '1'

rv = Path('/content/ai_influencer/rina_park/models/checkpoints/RealVisXL_V5.0_fp16.safetensors')
lora = Path('/content/ai_influencer/rina_park/models/loras/rina_park_person_sdxl_lora.safetensors')
print('RealVis:', rv.exists(), rv)
print('character LoRA:', lora.exists(), lora)
if rv.exists():
    !PYTHONPATH=/content/ai_influencer:/content/ai_influencer/rina_park TRANSFORMERS_NO_TF=1 USE_TF=0 USE_TORCH=1 python rina_park/scripts/generate_ig_quality.py
else:
    print('Skip smoke — download sdxl tier first')